#**Data Preprocessing**

In [ ]:
# imports
import os
import zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import gdown
import torch
import matplotlib.pyplot as plt

In [ ]:
# set random seed
random_seed = 67

## Importing Datasets

In [ ]:
from google.colab import userdata
!export KAGGLE_API_TOKEN={userdata.get('KAGGLE_API_TOKEN')}

# obtaining ds1 and ds2
import kagglehub
ds1 = kagglehub.dataset_download("linabennaa/eye-disease-image-dataset-mendeley")
ds2 = kagglehub.dataset_download("jeftaadriel/oia-odir-dataset")

100%|██████████| 2.13G/2.13G [01:34<00:00, 24.2MB/s]

Extracting files...


100%|██████████| 1.59G/1.59G [01:06<00:00, 25.5MB/s]

Extracting files...


### Dataset 1
https://data.mendeley.com/datasets/s9bfhswzjb/1

https://www.kaggle.com/datasets/linabennaa/eye-disease-image-dataset-mendeley

- Coverage - Retinitis Pigmentosa, Retinal Detachment, Pterygium, Myopia, Macular Scar, Glaucoma, Disc Edema, Diabetic Retinopathy, Central Serous Chorioretinopathy
- Images - 16242 (# patients not confirmed)

In [ ]:
dataset_path = ds1 + '/Augmented Dataset'

image_paths = []
disease_labels = []

# Iterate through each disease folder
for disease_folder in os.listdir(dataset_path):
  disease_path = os.path.join(dataset_path, disease_folder)

  # Iterate through images in the disease folder
  for image_name in os.listdir(disease_path):

    image_full_path = os.path.join(disease_path, image_name)
    image_paths.append(image_full_path)
    disease_labels.append(disease_folder)

# Create a DataFrame
df_ds1 = pd.DataFrame({
  'image_path': image_paths,
  'eye_disease': disease_labels,
  'SOURCE':'ds1'
})

### Data Set 2

https://www.kaggle.com/datasets/jeftaadriel/oia-odir-dataset

- NOTE: Labels are in Chinese and includes images for both eyes
- Coverage - DIABETIC_RETINOPATHY, GLAUCOMA, CATARACT, AGE_RELATED_MACULAR_DEGENERATION, HYPERTENSION, MYOPIA, OTHER_DISEASES
- Images - 13000 (10k train, 2k on site test, 1k off site test) (# patients not confirmed)

Since the rest of our datasets are for singular eyes, we will be splitting this dataset into left and right eye and classifying based on the diagnosis for each eye.

In [ ]:
# Convert files to csv format
off_site_file = pd.read_excel(f"{ds2}/Off-site Test Set/Annotation/off-site test annotation (English).xlsx")
off_site_file.to_csv("off_site_test_annotation.csv", index=None, header=True)

on_site_file = pd.read_excel(f"{ds2}/On-site Test Set/Annotation/on-site test annotation (English).xlsx")
on_site_file.to_csv("on_site_test_annotation.csv", index=None, header=True)

training_file = pd.read_excel(f"{ds2}/Training Set/Annotation/training annotation (English).xlsx")
training_file.to_csv("training_annotation.csv", index=None, header=True)

# Base paths for images
base_path_off_site = ds2 + '/Off-site Test Set/Images/'
base_path_on_site = ds2 + '/On-site Test Set/Images/'
base_path_training = ds2 + '/Training Set/Images/'

# Add image paths to a DataFrame
def process_fundus_image_paths(df, base_path, left_col='Left-Fundus', right_col='Right-Fundus'):
    def add_image_path(filename, current_base_path):
        if pd.isna(filename):
            return filename
        return os.path.join(current_base_path, filename)

    df[left_col] = df[left_col].apply(lambda x: add_image_path(x, base_path))
    df[right_col] = df[right_col].apply(lambda x: add_image_path(x, base_path))
    return df

# Apply to each DataFrame
off_site_file = process_fundus_image_paths(off_site_file, base_path_off_site)
on_site_file = process_fundus_image_paths(on_site_file, base_path_on_site)
training_file = process_fundus_image_paths(training_file, base_path_training)

# Combine all DataFrames into one
df_ds2 = pd.concat([off_site_file, on_site_file, training_file], ignore_index=True)

# Remove specified columns
columns_to_remove = ['ID', 'Patient Age', 'Patient Sex', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
df_ds2 = df_ds2.drop(columns=columns_to_remove)

# Convert Keywords into diagnosis for each eye
disease_mapping = {
    'normal': 'NORMAL',
    'diabetic retinopathy': 'DIABETIC_RETINOPATHY',
    'glaucoma': 'GLAUCOMA',
    'cataract': 'CATARACT',
    'macular degeneration': 'AGE_RELATED_MACULAR_DEGENERATION',
    'age-related macular degeneration': 'AGE_RELATED_MACULAR_DEGENERATION',
    'hypertensive retinopathy': 'HYPERTENSION',
    'myopia': 'MYOPIA',
}

df_left = df_ds2[['Left-Fundus', 'Left-Diagnostic Keywords']].copy()
df_right = df_ds2[['Right-Fundus', 'Right-Diagnostic Keywords']].copy()

df_left.columns = ['image_path', 'eye_disease']
df_right.columns = ['image_path', 'eye_disease']

# Get unique disease codes for column names
disease_codes = list(set(disease_mapping.values()))
disease_codes.append('OTHER_DISEASES')

def parse_diagnosis(df):
    # Initialize all disease columns with 0
    for disease in disease_codes:
        df[disease] = 0

    # Parse each row's diagnostic keywords
    for idx, keywords in df['eye_disease'].items():
        if pd.isna(keywords):
            continue

        # Convert to lowercase for matching
        keywords_lower = str(keywords).lower()

        disease_found = False

        # Check for each disease keyword in the mapping
        for keyword, disease_code in disease_mapping.items():
            if keyword in keywords_lower:
                df.at[idx, disease_code] = 1
                disease_found = True

        if not disease_found:
            df.at[idx, 'OTHER_DISEASES'] = 1

    df = df.drop('eye_disease', axis=1)

    return df

# Apply to both dataframes
df_left = parse_diagnosis(df_left)
df_right = parse_diagnosis(df_right)

# Combine to final dataset
df_ds2 = pd.concat([df_left, df_right], ignore_index=True)
df_ds2['SOURCE'] = 'ds2'

### Data Set 5

https://universe.roboflow.com/asmaa-qubdi/eye-diseases-7ng7m

- NOTE: The labels are in the png names…
- Coverage: Central Serous Chorioretinopathy, Diabetic Retinopathy, disc edema, glaucoma, macular scar, myopia, pterygium, retinal detachment, retinitis pigmentosa
- Images - 8425 images

In [ ]:
folder_link = "https://drive.google.com/drive/folders/1uoklEveAWV-yiaG9bWSv69jskgh_ySCs?usp=sharing"

download_folder = "/content/gdrive_public"
os.makedirs(download_folder, exist_ok=True)

result = gdown.download_folder(
	url = folder_link,
	output = download_folder,
	quiet = False,
	use_cookies = False
)

# get file in result
zip_path = next(f for f in result if f.endswith(".zip"))

extract_to = os.path.splitext(zip_path)[0]
os.makedirs(extract_to, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
  zf.extractall(extract_to)

# extracting all jpg files and putting them into one folder
import shutil

source = Path("/content/gdrive_public/Eye diseases.v1i.folder")
target = Path("/content/all_jpgs")
target.mkdir(parents=True, exist_ok=True)

jpg_files = list(source.rglob("*.jpg"))
print("Total images:", len(jpg_files))

for img_path in jpg_files:
  shutil.copy2(img_path, target / img_path.name)
# stored all of the jpg paths into target

# extracting classification
class_map = {
    "CSC": "Central Serous Chorioretinopathy",
    "DR": "Diabetic Retinopathy",
    "Disc-Edema": "Disc Edema",
    "Glaucoma": "Glaucoma",
    "Healthy": "Healthy",
    "Macular-Scar": "Macular Scar",
    "Myopia": "Myopia",
    "Pterygium": "Pterygium",
    "Retinal-Detachment": "Retinal Detachment",
    "Retinitis-Pigmentosa": "Retinitis Pigmentosa",
}

rows = []

for img_path in target.glob("*.jpg"):
  filename = img_path.name

  label = None
  for prefix, class_name in class_map.items():
    if filename.startswith(prefix):
      label = class_name
      break

  rows.append({
      "jpg_path": str(img_path),
      "classification": label
  })

df_ds5 = pd.DataFrame(rows)
df_ds5['SOURCE'] = 'ds5'

Retrieving folder contents


Processing file 1ejOBSpqz9rzkrcdKoG0DkJtbGFglN9UR Eye diseases.v1i.folder.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1ejOBSpqz9rzkrcdKoG0DkJtbGFglN9UR
From (redirected): https://drive.google.com/uc?id=1ejOBSpqz9rzkrcdKoG0DkJtbGFglN9UR&confirm=t&uuid=1b8f946e-6859-4017-b44c-817f226de96e
To: /content/gdrive_public/Eye diseases.v1i.folder.zip
100%|██████████| 1.40G/1.40G [00:22<00:00, 61.9MB/s]
Download completed


Total images: 9160


### Data Set 6

https://github.com/openmedlab/Awesome-Medical-Dataset/blob/main/resources/Multi-LabelRetinalDiseases.md

- Coverage: Diabetic Retinopathy, media haze, optic disc cupping, tessellation, age-related macular degeneration, drusen, myopia, branch retinal vein occlusion, optic disc pallor, central retinal vein occlusion, choroidal neovascularization, retinitis, optic disc edema, laser scars, central serious retinopathy, hypertensive retinopathy, arteriosclerotic retinopathy, chorioretinitis, other diseases
- Images - 2208 images
- Resolution types - ranging from 520x520 to 3400x2800

In [ ]:
folder_link_6 = "https://drive.google.com/drive/folders/1NO7LASTtbYvfE6-2lKGp2rAJXiSISHee?usp=sharing"

download_folder_6 = "/content/gdrive_public"
os.makedirs(download_folder_6, exist_ok=True)

result_6 = gdown.download_folder(
	url = folder_link_6,
	output = download_folder_6,
	quiet = False,
	use_cookies = False
)

# save the files separately
images_zip, train_data_csv, val_data_csv = result_6

# unzip images
extract_dir = "/content/gdrive_public/images"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(images_zip, "r") as zip_ref:
  zip_ref.extractall(extract_dir)

# checking the images within extract_dir
all_images = list(Path(extract_dir).rglob("*"))

Retrieving folder contents


Processing file 11nJZrKAhepO_zZ5Ef15QADsIG39y4lzZ images.zip
Processing file 1xtDYMkkr20Xm4FDxV4FHIhrJ8-A-tImm train_data_8.csv
Processing file 1lYk5996KZ7DEHDirKlrbWHoTvkvt2ix- val_data_8.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=11nJZrKAhepO_zZ5Ef15QADsIG39y4lzZ
From (redirected): https://drive.google.com/uc?id=11nJZrKAhepO_zZ5Ef15QADsIG39y4lzZ&confirm=t&uuid=05dbd84f-00dd-4079-9eae-29e10fd65413
To: /content/gdrive_public/images.zip
100%|██████████| 5.44G/5.44G [01:28<00:00, 61.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1xtDYMkkr20Xm4FDxV4FHIhrJ8-A-tImm
To: /content/gdrive_public/train_data_8.csv
100%|██████████| 80.1k/80.1k [00:00<00:00, 6.95MB/s]
Downloading...
From: https://drive.google.com/uc?id=1lYk5996KZ7DEHDirKlrbWHoTvkvt2ix-
To: /content/gdrive_public/val_data_8.csv
100%|██████████| 20.1k/20.1k [00:00<00:00, 24.2MB/s]
Download completed


In [ ]:
# combine the train and validation set into one
temp_df = pd.concat([pd.read_csv(train_data_csv), pd.read_csv(val_data_csv)], ignore_index=True)

# adding image path to temp_df
image_path_map = {
    Path(p).stem: str(p)
    for p in all_images
}

df_ds6 = temp_df.copy()
df_ds6["image_path"] = df_ds6["ID"].astype(str).map(image_path_map)

# remove ID column from df_ds6
df_ds6.drop(columns=["ID"], inplace=True)
df_ds6['SOURCE'] = 'ds6'

## Combining Datasets

In [ ]:
df_ds1_encoded = pd.get_dummies(df_ds1,columns = ['eye_disease'], prefix="", prefix_sep="", dtype=int)
df_ds1_encoded = df_ds1_encoded.rename(columns = {'Central Serous Chorioretinopathy_Color Fundus': 'Central Serous Retinopathy', 'Disc Edema': 'Optic Disc Edema','Healthy':'normal' })
df_ds1_encoded.columns = df_ds1_encoded.columns.str.upper()
df_ds1_encoded.columns = df_ds1_encoded.columns.str.replace(' ', '_')

In [ ]:
df_ds2_encoded = df_ds2.drop(columns=["OTHER_DISEASES"])
df_ds2_encoded.columns = df_ds2_encoded.columns.str.upper()

In [ ]:
df_ds5_encoded = pd.get_dummies(df_ds5, columns=['classification'], prefix="", prefix_sep="", dtype=int)
df_ds5_encoded = df_ds5_encoded.rename(columns={'Healthy': 'NORMAL', 'jpg_path': 'image_path'})
df_ds5_encoded.columns = df_ds5_encoded.columns.str.upper()
df_ds5_encoded.columns = df_ds5_encoded.columns.str.replace(' ', '_')

In [ ]:
df_ds6_encoded = df_ds6.rename(columns={'DR': 'DIABETIC_RETINOPATHY', 'MH':'MEDIA_HAZE', 'ODC': 'OPTIC_DISC_CUPPING', 'TSLN':'TESSELLATION', 'ARMD':'AGE_RELATED_MACULAR_DEGENERATION', 'DN':'DRUSEN', 'MYA':'MYOPIA', 'BRVO':'BRANCH_RETINAL_VEIN_OCCLUSION', 'ODP':'OPTIC_DISC_PALLOR', 'CRVO':'CENTRAL_RETINAL_VEIN_OCCLUSION', 'CNV':'CHOROIDAL NEOVASCULARIZATION', 'RS':'RETINITIS', 'ODE':'OPTIC_DISC_EDEMA', 'LS':'LASER_SCARS', 'CSR':'CENTRAL_SEROUS_RETINOPATHY', 'HTR':'HYPERTENSIVE_RETINOPATHY', 'ASR':'ARTERIOSCLEROTIC_RETINOPATHY', 'CRS':'CHORIORETINITIS'})
df_ds6_encoded = df_ds6_encoded.drop(columns=['OTHER'])
df_ds6_encoded.columns = df_ds6_encoded.columns.str.upper()
df_ds6_encoded.columns = df_ds6_encoded.columns.str.replace(' ', '_')

In [ ]:
df_full_encoded = pd.concat([df_ds1_encoded, df_ds2_encoded, df_ds5_encoded, df_ds6_encoded], ignore_index=True).fillna(0)
float_cols = df_full_encoded.select_dtypes(include="float").columns
df_full_encoded[float_cols] = df_full_encoded[float_cols].astype(int)

### Removing repeated eye diseases

In [ ]:
# Make a copy of the full encoded dataframe (assuming df_full_encoded exists)
df_merged = df_full_encoded.copy()

# Merge: CENTRAL_SEROUS_RETINOPATHY + CENTRAL_SEROUS_CHORIORETINOPATHY
df_merged['CENTRAL_SEROUS_RETINOPATHY'] = (
    df_merged['CENTRAL_SEROUS_RETINOPATHY'] | df_merged['CENTRAL_SEROUS_CHORIORETINOPATHY']
).astype(int)
df_merged = df_merged.drop(columns=['CENTRAL_SEROUS_CHORIORETINOPATHY'])

# Merge: OPTIC_DISC_EDEMA + DISC_EDEMA
df_merged['DISC_EDEMA'] = (
    df_merged['OPTIC_DISC_EDEMA'] | df_merged['DISC_EDEMA']
).astype(int)
df_merged = df_merged.drop(columns=['OPTIC_DISC_EDEMA'])

# Merge: HYPERTENSION + HYPERTENSIVE_RETINOPATHY
df_merged['HYPERTENSION'] = (
    df_merged['HYPERTENSION'] | df_merged['HYPERTENSIVE_RETINOPATHY']
).astype(int)
df_merged = df_merged.drop(columns=['HYPERTENSIVE_RETINOPATHY'])

# Merge: RETINITIS + CHORIORETINITIS
df_merged['RETINITIS'] = (
    df_merged['RETINITIS'] | df_merged['CHORIORETINITIS']
).astype(int)
df_merged = df_merged.drop(columns=['CHORIORETINITIS'])

# Update label_cols_merged after merging
label_cols_merged = [
    'CENTRAL_SEROUS_RETINOPATHY',
    'DIABETIC_RETINOPATHY',
    'GLAUCOMA',
    'NORMAL',
    'MACULAR_SCAR',
    'MYOPIA',
    'PTERYGIUM',
    'RETINAL_DETACHMENT',
    'RETINITIS_PIGMENTOSA',
    'HYPERTENSION',
    'CATARACT',
    'AGE_RELATED_MACULAR_DEGENERATION',
    'DISC_EDEMA',
    'MEDIA_HAZE',
    'OPTIC_DISC_CUPPING',
    'TESSELLATION',
    'DRUSEN',
    'BRANCH_RETINAL_VEIN_OCCLUSION',
    'OPTIC_DISC_PALLOR',
    'CENTRAL_RETINAL_VEIN_OCCLUSION',
    'CHOROIDAL_NEOVASCULARIZATION',
    'RETINITIS',
    'LASER_SCARS',
    'ARTERIOSCLEROTIC_RETINOPATHY',
]

'''
# Ensure merged class names are in the list
if 'CENTRAL_SEROUS_RETINOPATHY' not in label_cols_merged:
    label_cols_merged.append('CENTRAL_SEROUS_RETINOPATHY')
if 'OPTIC_DISC_EDEMA' not in label_cols_merged:
    label_cols_merged.append('OPTIC_DISC_EDEMA')
if 'HYPERTENSION' not in label_cols_merged:
    label_cols_merged.append('HYPERTENSION')
if 'RETINITIS' not in label_cols_merged:
    label_cols_merged.append('RETINITIS')'''

print("Merged label columns:", label_cols_merged)

Merged label columns: ['CENTRAL_SEROUS_RETINOPATHY', 'DIABETIC_RETINOPATHY', 'GLAUCOMA', 'NORMAL', 'MACULAR_SCAR', 'MYOPIA', 'PTERYGIUM', 'RETINAL_DETACHMENT', 'RETINITIS_PIGMENTOSA', 'HYPERTENSION', 'CATARACT', 'AGE_RELATED_MACULAR_DEGENERATION', 'DISC_EDEMA', 'MEDIA_HAZE', 'OPTIC_DISC_CUPPING', 'TESSELLATION', 'DRUSEN', 'BRANCH_RETINAL_VEIN_OCCLUSION', 'OPTIC_DISC_PALLOR', 'CENTRAL_RETINAL_VEIN_OCCLUSION', 'CHOROIDAL_NEOVASCULARIZATION', 'RETINITIS', 'LASER_SCARS', 'ARTERIOSCLEROTIC_RETINOPATHY']


# Image Preprocessing

## Image Transformation & Augmentation

In [ ]:
# crop black borders of the images
def crop_black_borders(image):
    # Convert PIL Image to NumPy array
    img_np = np.array(image)

    # Threshold for black pixels
    # Pixels with R, G, B values all below this threshold will be considered black
    BLACK_THRESHOLD = 10

    non_black_pixels = np.any(img_np > BLACK_THRESHOLD, axis=2)

    # Find rows and columns that contain non-black pixels
    rows = np.any(non_black_pixels, axis=1)
    cols = np.any(non_black_pixels, axis=0)

    # Get the min/max indices for cropping
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]

    # Crop the NumPy array
    cropped_img_np = img_np[rmin:rmax+1, cmin:cmax+1]

    # Convert back to PIL Image
    return Image.fromarray(cropped_img_np)

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

class FundusMultiLabelDataset(Dataset):
  def __init__(self, df:pd.DataFrame, label_cols, transform=None):
    self.labels_df = df
    self.transform = transform
    self.label_cols = label_cols

  def __len__(self):
    return len(self.labels_df)

  def __getitem__(self, idx):
    image_row = self.labels_df.iloc[idx]
    image_path = image_row['IMAGE_PATH']
    image = Image.open(image_path)

    if image.mode == "I;16":
      arr = np.array(image).astype("float32") / 65535.0
      arr = (arr * 255).astype("uint8")
      image = Image.fromarray(arr)

    image = image.convert("RGB")

    if self.transform:
      image = self.transform(image)

    labels = torch.tensor(image_row[self.label_cols].values.astype(np.float32), dtype=torch.float32)

    return {"image": image, "labels": labels}

### Transformation metrics

In [ ]:
# define transformations
train_transform = v2.Compose([
  v2.Lambda(crop_black_borders),
  v2.ToImage(),
  v2.Resize((224,224), antialias=True),
  v2.RandomHorizontalFlip(p=0.5),
  v2.RandomVerticalFlip(p=0.5),
  v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
  v2.RandomApply([v2.GaussianBlur(kernel_size=3, sigma=(0.1,1.0))], p=0.3),
  v2.ToDtype(torch.float32, scale=True),
  v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = v2.Compose([
  v2.Lambda(crop_black_borders),
  v2.ToImage(),
  v2.Resize((224,224), antialias=True),
  v2.ToDtype(torch.float32, scale=True),
  v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

### Additional augmentation for small classes

In [ ]:
# Identify classes with < 100 images and augment them

# Count images per class (after merging)
class_counts = df_merged[label_cols_merged].sum()
low_freq_classes = class_counts[class_counts < 100].index.tolist()
print(f"Low-frequency classes (<100 images): {low_freq_classes}")

# Define augmentation functions with randomized Gaussian blur (sigma 0-0.1)
def augment_with_blur(image, seed_offset=0, flip_horizontal=False, flip_vertical=False):
    """
    Apply flips and a small randomized Gaussian blur.
    seed_offset ensures different random blur for each augmentation.
    """
    # Set seed for reproducibility based on seed_offset
    torch.manual_seed(67 + seed_offset)

    # Apply flips
    if flip_horizontal:
        image = v2.functional.horizontal_flip(image)
    if flip_vertical:
        image = v2.functional.vertical_flip(image)

    # Apply small randomized Gaussian blur (sigma between 0 and 0.1)
    # Note: GaussianBlur sigma expects at least 0.1, so we use a small range
    blur_sigma = (0.01, 0.1)  # Small blur amount
    blur = v2.GaussianBlur(kernel_size=3, sigma=blur_sigma)
    image = blur(image)

    return image

# Prepare list to collect augmented rows
augmented_rows = []

# For each low-frequency class, augment ALL images in that class
for cls in low_freq_classes:
    # Get indices where this class is 1
    cls_indices = df_merged[df_merged[cls] == 1].index.tolist()

    for idx in cls_indices:
        original_row = df_merged.loc[idx].to_dict()
        image_path = original_row['IMAGE_PATH']

        # Load image once
        img = Image.open(image_path)
        if img.mode == "I;16":
            arr = np.array(img).astype("float32") / 65535.0
            arr = (arr * 255).astype("uint8")
            img = Image.fromarray(arr)
        img = img.convert("RGB")

        # Convert to tensor for augmentation
        img_tensor = v2.ToImage()(img)

        # Create 3 augmented versions:
        # 1. Horizontal flip + small Gaussian blur
        img_aug1 = augment_with_blur(img_tensor, seed_offset=1,
                                      flip_horizontal=True, flip_vertical=False)
        # 2. Vertical flip + small Gaussian blur
        img_aug2 = augment_with_blur(img_tensor, seed_offset=2,
                                      flip_horizontal=False, flip_vertical=True)
        # 3. Horizontal + Vertical flip + small Gaussian blur
        img_aug3 = augment_with_blur(img_tensor, seed_offset=3,
                                      flip_horizontal=True, flip_vertical=True)

        # Save augmented images and add rows
        aug_configs = [
            ('hflip_blur', img_aug1),
            ('vflip_blur', img_aug2),
            ('hvflip_blur', img_aug3)
        ]

        for aug_name, aug_img in aug_configs:
            # Convert back to PIL for saving
            aug_pil = v2.ToPILImage()(aug_img)
            # Create new path
            base, ext = os.path.splitext(image_path)
            new_path = f"{base}_{aug_name}{ext}"
            aug_pil.save(new_path)

            # Create new row with same labels
            new_row = original_row.copy()
            new_row['IMAGE_PATH'] = new_path
            new_row['SOURCE'] = original_row['SOURCE'] + f"_aug_{aug_name}"
            augmented_rows.append(new_row)

# Convert augmented rows to DataFrame and concatenate
df_augmented = pd.DataFrame(augmented_rows)
df_final = pd.concat([df_merged, df_augmented], ignore_index=True)

print(f"Original size: {len(df_merged)}")
print(f"After augmentation: {len(df_final)}")
print(f"Expected size: {len(df_merged)} + {len(df_merged[df_merged[low_freq_classes].any(axis=1)]) * 3}")

Low-frequency classes (<100 images): ['BRANCH_RETINAL_VEIN_OCCLUSION', 'OPTIC_DISC_PALLOR', 'CENTRAL_RETINAL_VEIN_OCCLUSION', 'CHOROIDAL_NEOVASCULARIZATION', 'RETINITIS', 'LASER_SCARS', 'ARTERIOSCLEROTIC_RETINOPATHY']
Original size: 37610
After augmentation: 38879
Expected size: 37610 + 1200


### All classes

In [ ]:
LABEL_COLS = label_cols_merged

# **Train/Val/Test Split**

In [ ]:
!pip install iterative-stratification

## Initial multi-label stratification

In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

def multilabel_train_val_test_split(df, label_cols, random_state, train_size=0.8, val_size=0.1, test_size=0.1):
  y = df[label_cols].values

  msss1 = MultilabelStratifiedShuffleSplit(
      n_splits=1,
      test_size=(1-train_size),
      random_state=random_state)

  train_idx, temp_idx = next(msss1.split(df,y))

  train_df = df.iloc[train_idx].reset_index(drop=True)
  temp_df = df.iloc[temp_idx].reset_index(drop=True)

  y_temp = temp_df[label_cols].values

  val_ratio = val_size / (val_size + test_size)

  msss2 = MultilabelStratifiedShuffleSplit(
      n_splits=1,
      test_size=(1-val_ratio),
      random_state=random_state)

  val_idx, test_idx = next(msss2.split(temp_df, y_temp))

  val_df = temp_df.iloc[val_idx].reset_index(drop=True)
  test_df = temp_df.iloc[test_idx].reset_index(drop=True)
  return train_df, val_df, test_df

In [ ]:
# conduct the split
train_df, val_df, test_df = multilabel_train_val_test_split(
    df_final,
    LABEL_COLS,
    random_state=random_seed)

In [ ]:
# obtain the dataset (transformed data)
train_dataset = FundusMultiLabelDataset(
    df=train_df,
    label_cols=LABEL_COLS,
    transform=train_transform
)

val_dataset = FundusMultiLabelDataset(
    df=val_df,
    label_cols=LABEL_COLS,
    transform=val_transform
)

test_dataset = FundusMultiLabelDataset(
    df=test_df,
    label_cols=LABEL_COLS,
    transform=val_transform
)

In [ ]:
# DataLoader for the CNNs
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
    )

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

## Multi-label stratification for the CNNs

In [ ]:
# Multilabel stratified split (80/20) for training/validation (for the CNNs)
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

def multilabel_train_val_split(df, label_cols, random_state=67, train_size=0.8):
    y = df[label_cols].values

    msss = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=(1 - train_size),
        random_state=random_state
    )

    train_idx, val_idx = next(msss.split(df, y))

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    return train_df, val_df

# Perform the split
train_df_final, val_df_final = multilabel_train_val_split(
    train_df,
    label_cols_merged,
    random_state=67,
    train_size=0.8
)

print(f"Train size: {len(train_df_final)}")
print(f"Validation size: {len(val_df_final)}")

# Create Datasets and DataLoaders (for the CNNs)
train_dataset_final = FundusMultiLabelDataset(
    df=train_df_final,
    label_cols=LABEL_COLS,
    transform=train_transform
)

val_dataset_final = FundusMultiLabelDataset(
    df=val_df_final,
    label_cols=LABEL_COLS,
    transform=val_transform
)

train_loader_final = DataLoader(
    train_dataset_final,
    batch_size=32,
    shuffle=True,  # Shuffle for training
    num_workers=2,
    pin_memory=True
)

val_loader_final = DataLoader(
    val_dataset_final,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

Train size: 24882
Validation size: 6221


# **ShuffleNet**

In [ ]:
# imports
import torch
import torch.nn as nn
import numpy as np
import torch.optim as optim
import random
from torchvision import models
from torchvision.models import ShuffleNet_V2_X1_0_Weights
from sklearn.metrics import f1_score, roc_auc_score
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

def set_seed(seed=67):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(67)

cuda


In [ ]:
!nvidia-smi

Sun Apr 19 15:53:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    a = torch.randn(2000, 2000, device="cuda")
    b = torch.randn(2000, 2000, device="cuda")
    c = a @ b
    print("Result device:", c.device)

CUDA available: True
Result device: cuda:0


## Defining the model

In [ ]:
class ShuffleNetV2MultiLabel(nn.Module):
    def __init__(self, num_classes, pretrained=True, dropout=0.2, return_features=False):
        super().__init__()
        self.return_features = return_features

        weights = ShuffleNet_V2_X1_0_Weights.DEFAULT if pretrained else None
        backbone = models.shufflenet_v2_x1_0(weights=weights)

        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity() # remove original classifier

        self.backbone = backbone
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x):
        features = self.backbone(x) # [B, 1024]
        logits = self.classifier(self.dropout(features))

        if self.return_features:
            return logits, features
        return logits

## Creating the model

In [ ]:
num_classes = len(LABEL_COLS)

model = ShuffleNetV2MultiLabel(
    num_classes=num_classes,
    pretrained=True,
    dropout=0.2,
    return_features=False
).to(device)

print(model)

Downloading: "https://download.pytorch.org/models/shufflenetv2_x1-5666bf0f80.pth" to /root/.cache/torch/hub/checkpoints/shufflenetv2_x1-5666bf0f80.pth


100%|██████████| 8.79M/8.79M [00:00<00:00, 126MB/s]

ShuffleNetV2MultiLabel(
  (backbone): ShuffleNetV2(
    (conv1): Sequential(
      (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (stage2): Sequential(
      (0): InvertedResidual(
        (branch1): Sequential(
          (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=24, bias=False)
          (1): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): Conv2d(24, 58, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (3): BatchNorm2d(58, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (4): ReLU(inplace=True)
        )
        (branch2): Sequential(
          (0): Conv2d(24, 58, kernel_size=(1, 1), stride=(1, 1), bias=False)
         

## Loss, optimizer, and scheduler

In [ ]:
label_counts = train_df_final[LABEL_COLS].sum(axis=0).values
num_samples = len(train_df_final)

pos_weight = (num_samples - label_counts) / np.clip(label_counts, a_min=1, a_max=None)
pos_weight = torch.tensor(pos_weight, dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.Adam(model.parameters(), lr=1e-4)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.1
)

## Training function

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for batch in tqdm(loader, desc="Training", leave=False):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss

## Validation function

In [ ]:
@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, threshold=0.5):
    model.eval()
    running_loss = 0.0

    all_labels = []
    all_probs = []

    for batch in tqdm(loader, desc="Validation", leave=False):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        probs = torch.sigmoid(logits)

        running_loss += loss.item() * images.size(0)
        all_labels.append(labels.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    y_true = np.vstack(all_labels)
    y_prob = np.vstack(all_probs)
    y_pred = (y_prob >= threshold).astype(int)

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)

    try:
        macro_auc = roc_auc_score(y_true, y_prob, average="macro")
    except ValueError:
        macro_auc = float("nan")

    return epoch_loss, macro_f1, micro_f1, macro_auc

## Training using CNN split of training/val data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import os

os.makedirs("/content/drive/MyDrive/shufflenet", exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
os.makedirs("/content/drive/MyDrive/shufflenet", exist_ok=True)

In [ ]:
# also save locally
save_local_model = "/content/LOCAL_shufflenetv2_model_feature.pth"
save_local_weights = "/content/LOCAL_weights_shufflenetv2_model_feature.pth"

In [2]:
from google.colab import files

In [ ]:
num_epochs = 15
best_val_f1 = -1.0
save_path_model = "/content/drive/MyDrive/shufflenet/shufflenetv2_model_feature.pth"
save_path_weights = "/content/drive/MyDrive/shufflenet/weights_shufflenetv2_model_feature.pth"

In [42]:
history = {
    "train_loss": [],
    "val_loss": [],
    "val_macro_f1": [],
    "val_micro_f1": [],
    "val_macro_auc": []
}

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader_final,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    val_loss, val_macro_f1, val_micro_f1, val_macro_auc = validate_one_epoch(
        model=model,
        loader=val_loader_final,
        criterion=criterion,
        device=device,
        threshold=0.5
    )

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_macro_f1"].append(val_macro_f1)
    history["val_micro_f1"].append(val_micro_f1)
    history["val_macro_auc"].append(val_macro_auc)

    print(f"Train Loss:   {train_loss:.4f}")
    print(f"Val Loss:     {val_loss:.4f}")
    print(f"Val Macro F1: {val_macro_f1:.4f}")
    print(f"Val Micro F1: {val_micro_f1:.4f}")
    print(f"Val Macro AUC:{val_macro_auc:.4f}")

    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        torch.save(model, save_path_model)
        torch.save(model, save_local_model)
        print(f"Saved best model to {save_path_model}")

        torch.save(model.state_dict(), save_path_weights)
        torch.save(model.state_dict(), save_local_weights)
        print(f"Saved best model weights to {save_path_weights}")


Epoch [1/15]


Training:   0%|          | 0/778 [00:00<?, ?it/s]

Validation:   0%|          | 0/195 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e08b1c5e2a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e08b1c5e2a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Train Loss:   0.6021
Val Loss:     0.4437
Val Macro F1: 0.2890
Val Micro F1: 0.3268
Val Macro AUC:0.9512
Saved best model to /content/drive/MyDrive/shufflenet/shufflenetv2_model_feature.pth
Saved best model weights to /content/drive/MyDrive/shufflenet/weights_shufflenetv2_model_feature.pth

Epoch [2/15]


...

Train Loss:   0.2482
Val Loss:     0.2356
Val Macro F1: 0.4587
Val Micro F1: 0.5355
Val Macro AUC:0.9807
Saved best model to /content/drive/MyDrive/shufflenet/shufflenetv2_model_feature.pth
Saved best model weights to /content/drive/MyDrive/shufflenet/weights_shufflenetv2_model_feature.pth

Epoch [15/15]


Training:   0%|          | 0/778 [00:00<?, ?it/s]

Validation:   0%|          | 0/195 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e08b1c5e2a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e08b1c5e2a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Train Loss:   0.2482
Val Loss:     0.2339
Val Macro F1: 0.4542
Val Micro F1: 0.5425
Val Macro AUC:0.9809


In [3]:
files.download("/content/LOCAL_shufflenetv2_model_feature.pth")
files.download("/content/LOCAL_weights_shufflenetv2_model_feature.pth")

FileNotFoundError: Cannot find file: /content/LOCAL_shufflenetv2_model_feature.pth

In [ ]:
from google.colab import runtime

# save checkpoints / outputs first
runtime.unassign()